# **6. Recomendador híbrido**

In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

In [2]:
# Información descriptiva de las canciones
df = pd.read_csv('../data/spotify-tracks-dataset-clean.csv')

# Características musicales estandarizadas
X_audio_scaled = pd.read_csv('../data/spotify-audio-features-scaled.csv', index_col = 0)

print('Dimensiones de df:', df.shape)
print('Dimensiones de X_audio_scaled:', X_audio_scaled.shape)
print('¿Tienen los mismos índices?', df.index.equals(X_audio_scaled.index))

Dimensiones de df: (89578, 20)
Dimensiones de X_audio_scaled: (89578, 9)
¿Tienen los mismos índices? True


### **6.1 Géneros**

In [3]:
# Mostrar algunos ejemplos de canciones y sus géneros
display(df[['track_name', 'artists', 'track_genre']].sample(10, random_state = 123))

,track_name,artists,track_genre
88298,I Do,boom boom kid,punk;punk-rock
80586,Perfect Strangers,Jonas Blue;JP Cooper,edm;house
70897,A Number of Microphones,Propellerheads,trip-hop
56199,Amante Profissional,Herva Doce,r-n-b
62191,Haus of Holbein,SIX,show-tunes
26568,O Rosto de Cristo,Sarah Farias,gospel
62932,Her Or Me,Claude-Michel Schönberg;Claire Moore,show-tunes
86986,Palavras De Um Futuro Bom,Jota Quest,mpb;r-n-b
76376,Sossego,Tim Maia,brazil;mpb;samba;soul
84974,Love You Anymore,Michael Bublé,j-pop;jazz


In [4]:
# Número de géneros asociados a cada canción
numero_generos = df['track_genre'].str.split(';').str.len()

print('Número máximo de géneros:', numero_generos.max())
print('Distribución del número de géneros:')
display(numero_generos.value_counts().sort_index())

Número máximo de géneros: 9
Distribución del número de géneros:


track_genre
1    73280
2    11423
3     2955
4     1361
5      431
6      104
7       21
8        2
9        1
Name: count, dtype: int64

In [5]:
# Separar los géneros de cada canción y almacenarlos como conjuntos
df['genres_set'] = df['track_genre'].apply(
    lambda x: set(x.split(';'))
)

# Mostrar algunos ejemplos
display(df[['track_name', 'track_genre', 'genres_set']].sample(10, random_state = 123))

,track_name,track_genre,genres_set
88298,I Do,punk;punk-rock,"{punk, punk-rock}"
80586,Perfect Strangers,edm;house,"{house, edm}"
70897,A Number of Microphones,trip-hop,{trip-hop}
56199,Amante Profissional,r-n-b,{r-n-b}
62191,Haus of Holbein,show-tunes,{show-tunes}
26568,O Rosto de Cristo,gospel,{gospel}
62932,Her Or Me,show-tunes,{show-tunes}
86986,Palavras De Um Futuro Bom,mpb;r-n-b,"{mpb, r-n-b}"
76376,Sossego,brazil;mpb;samba;soul,"{mpb, brazil, soul, samba}"
84974,Love You Anymore,j-pop;jazz,"{jazz, j-pop}"


In [6]:
numero_generos = df['genres_set'].apply(len)

print('Número máximo de géneros:', numero_generos.max())
print('Distribución del número de géneros:')
display(numero_generos.value_counts().sort_index())

Número máximo de géneros: 9
Distribución del número de géneros:


genres_set
1    73280
2    11423
3     2955
4     1361
5      431
6      104
7       21
8        2
9        1
Name: count, dtype: int64

### **6.1.1 Filtro de géneros**

In [7]:
# Identificar la canción de referencia
indice = 57411

# Recuperar sus géneros
generos_referencia = df.loc[indice, 'genres_set']

print('Géneros de la canción de referencia:')
print(generos_referencia)

Géneros de la canción de referencia:
{'rock'}


In [8]:
# Identificar canciones que compartan al menos un género
mascara_genero = df['genres_set'].apply(
    lambda generos: bool(generos_referencia & generos)
)

# Excluir la canción de referencia
mascara_genero.loc[indice] = False

# Recuperar las canciones candidatas
candidatas = df.loc[mascara_genero].copy()

print('Número de canciones candidatas:', len(candidatas))

display(candidatas[['track_name', 'artists', 'track_genre']].head(10))

Número de canciones candidatas: 999


,track_name,artists,track_genre
57259,"Sex, Drugs, Etc.",Beach Weather,rock
57260,Thunder,Imagine Dragons,rock
57261,Believer,Imagine Dragons,rock
57262,Bones,Imagine Dragons,rock
57263,World's Smallest Violin,AJR,rock
57264,Whatever It Takes,Imagine Dragons,rock
57265,Demons,Imagine Dragons,rock
57266,Bad Liar,Imagine Dragons,rock
57267,House of Memories,Panic! At The Disco,rock
57268,Bones,Imagine Dragons,rock


### **6.2 Selección mediante distancia euclidiana**

In [9]:
# Recuperar las características musicales de la canción de referencia
caracteristicas_referencia = X_audio_scaled.loc[[indice]]

# Recuperar las características musicales de las canciones candidatas
caracteristicas_candidatas = X_audio_scaled.loc[candidatas.index]

# Mostrar dimensiones
print('Dimensiones del vector de referencia:', caracteristicas_referencia.shape)
print('Dimensiones de los vectores candidatos:', caracteristicas_candidatas.shape)

Dimensiones del vector de referencia: (1, 9)
Dimensiones de los vectores candidatos: (999, 9)


In [10]:
# Calcular las distancias euclidianas
distancias = euclidean_distances(caracteristicas_referencia, caracteristicas_candidatas).flatten()

# Asociar cada distancia con el índice de su canción
distancias = pd.Series(distancias, index = candidatas.index)

# Ordenar las canciones de menor a mayor distancia
distancias_ordenadas = distancias.sort_values()

# Mostrar las 10 canciones más cercanas
display(
    candidatas.loc[
        distancias_ordenadas.head(10).index, ['track_name', 'artists', 'track_genre']
    ].assign(distancia_euclidiana = distancias_ordenadas.head(10))
)

,track_name,artists,track_genre,distancia_euclidiana
83506,Rock And Roll All Nite,KISS,hard-rock;metal;rock,0.870047
83505,Rock And Roll All Nite,KISS,hard-rock;metal;rock,0.870047
57512,Fly Away,Lenny Kravitz,rock,1.273074
81453,My Songs Know What You Did In The Dark (Light ...,Fall Out Boy,emo;rock,1.587308
81452,My Songs Know What You Did In The Dark (Light ...,Fall Out Boy,emo;rock,1.587308
73688,In Over My Head,grandson,alt-rock;alternative;rock,1.588413
83511,Hell Or High Water,KISS,hard-rock;metal;rock,1.851981
57259,"Sex, Drugs, Etc.",Beach Weather,rock,1.890935
57300,Radioactive,Imagine Dragons,rock,2.077368
84616,Good Time,Owl City;Carly Rae Jepsen,indie;indie-pop;rock,2.111448


In [11]:
# Seleccionar las 100 canciones con menor distancia euclidiana
indices_seleccionados = distancias_ordenadas.head(100).index

# Recuperar sus características musicales
X_candidatas = X_audio_scaled.loc[indices_seleccionados]

# Recuperar sus metadatos
df_candidatas = candidatas.loc[indices_seleccionados]

print('Número de canciones seleccionadas:', len(df_candidatas))
print('Dimensiones de X_candidatas:', X_candidatas.shape)

Número de canciones seleccionadas: 100
Dimensiones de X_candidatas: (100, 9)


### **6.3 Similitud del coseno**

In [12]:
# Calcular la similitud del coseno
similitudes = cosine_similarity(caracteristicas_referencia, X_candidatas).flatten()

# Asociar cada similitud con el índice de su canción
similitudes = pd.Series(similitudes, index = X_candidatas.index)

# Ordenar las canciones de mayor a menor similitud
similitudes_ordenadas = similitudes.sort_values(ascending = False)

similitudes_ordenadas

83506    0.976500
83505    0.976500
57512    0.933966
81453    0.910548
81452    0.910548
           ...   
83484    0.520080
87505    0.515230
87456    0.515230
87502    0.515230
87454    0.515230
Length: 100, dtype: float64

In [13]:
# Construir la tabla de recomendaciones
recomendaciones = df_candidatas.loc[
    similitudes_ordenadas.index, ['track_name', 'artists', 'track_genre', 'album_name']
].copy()

# Incorporar las métricas de similitud
recomendaciones['distancia_euclidiana'] = distancias.loc[recomendaciones.index]

recomendaciones['similitud_coseno'] = similitudes_ordenadas

# Mostrar las 10 mejores recomendaciones
display(recomendaciones.head(10))

,track_name,artists,track_genre,album_name,distancia_euclidiana,similitud_coseno
83506,Rock And Roll All Nite,KISS,hard-rock;metal;rock,Rock Workout 2023,0.870047,0.976500
83505,Rock And Roll All Nite,KISS,hard-rock;metal;rock,Rock & Halloween,0.870047,0.976500
57512,Fly Away,Lenny Kravitz,rock,90's Music Grandi Successi,1.273074,0.933966
81453,My Songs Know What You Did In The Dark (Light ...,Fall Out Boy,emo;rock,Halloween Party 2022,1.587308,0.910548
81452,My Songs Know What You Did In The Dark (Light ...,Fall Out Boy,emo;rock,Halloween 2022,1.587308,0.910548
73688,In Over My Head,grandson,alt-rock;alternative;rock,20's Rock,1.588413,0.909358
57259,"Sex, Drugs, Etc.",Beach Weather,rock,Chit Chat,1.890935,0.874359
83511,Hell Or High Water,KISS,hard-rock;metal;rock,Rock & Halloween,1.851981,0.863101
57275,Follow You,Imagine Dragons,rock,pov: it's 2021,2.193395,0.837489
57466,Follow You,Imagine Dragons,rock,Follow You / Cutthroat,2.193395,0.837489


### **6.3.1 Tratamiento de las repeticiones**

In [14]:
# Identificar canciones repetidas por título y artista
repeticiones = recomendaciones[
    recomendaciones.duplicated(
        subset = ['track_name', 'artists'], keep = False
    )
].sort_values(
    by = ['track_name', 'artists']
)

display(repeticiones.head(10))

,track_name,artists,track_genre,album_name,distancia_euclidiana,similitud_coseno
88020,All The Small Things,blink-182,punk;punk-rock;rock,Halloween Party Canzoni per Bambini 2022,2.335188,0.762811
88019,All The Small Things,blink-182,punk;punk-rock;rock,Halloween Kids Party 2022,2.335188,0.762811
88023,All The Small Things,blink-182,punk;punk-rock;rock,Rock Workout 2023,2.335188,0.762811
88025,All The Small Things,blink-182,punk;punk-rock;rock,Mirando las nubes,2.335188,0.762811
88018,All The Small Things,blink-182,punk;punk-rock;rock,90's Music Grandi Successi,2.335188,0.762811
88008,All The Small Things,blink-182,punk;punk-rock;rock,OO's Music Grandi Successi,2.335188,0.762811
81986,Christmas All Over Again,Tom Petty and the Heartbreakers,folk;rock;singer-songwriter;songwriter,Rockin' Around The Christmas Tree 2022,2.138039,0.828829
81995,Christmas All Over Again,Tom Petty and the Heartbreakers,folk;rock;singer-songwriter;songwriter,Christmas Best Hits 2022,2.138039,0.828829
81994,Christmas All Over Again,Tom Petty and the Heartbreakers,folk;rock;singer-songwriter;songwriter,Santa Claus Is Coming To Town,2.138039,0.828829
81997,Christmas All Over Again,Tom Petty and the Heartbreakers,folk;rock;singer-songwriter;songwriter,Christmas Rock 2022,2.138039,0.828829


In [15]:
# Eliminar canciones repetidas por título y artista
recomendaciones_unicas = (
    recomendaciones
    .sort_values(
        by = ['similitud_coseno', 'distancia_euclidiana'],
        ascending = [False, True]
    )
    .drop_duplicates(
        subset = ['track_name', 'artists'],
        keep = 'first'
    )
)

# Mostrar las 10 mejores recomendaciones sin repeticiones
display(recomendaciones_unicas.head(10))

print('Recomendaciones originales:', len(recomendaciones))
print('Recomendaciones únicas:', len(recomendaciones_unicas))

,track_name,artists,track_genre,album_name,distancia_euclidiana,similitud_coseno
83506,Rock And Roll All Nite,KISS,hard-rock;metal;rock,Rock Workout 2023,0.870047,0.976500
57512,Fly Away,Lenny Kravitz,rock,90's Music Grandi Successi,1.273074,0.933966
81453,My Songs Know What You Did In The Dark (Light ...,Fall Out Boy,emo;rock,Halloween Party 2022,1.587308,0.910548
73688,In Over My Head,grandson,alt-rock;alternative;rock,20's Rock,1.588413,0.909358
57259,"Sex, Drugs, Etc.",Beach Weather,rock,Chit Chat,1.890935,0.874359
83511,Hell Or High Water,KISS,hard-rock;metal;rock,Rock & Halloween,1.851981,0.863101
57275,Follow You,Imagine Dragons,rock,pov: it's 2021,2.193395,0.837489
84616,Good Time,Owl City;Carly Rae Jepsen,indie;indie-pop;rock,Tutti in piscina Canzoni per bambini,2.111448,0.832613
81986,Christmas All Over Again,Tom Petty and the Heartbreakers,folk;rock;singer-songwriter;songwriter,Rockin' Around The Christmas Tree 2022,2.138039,0.828829
57300,Radioactive,Imagine Dragons,rock,Night Visions,2.077368,0.811069


Recomendaciones originales: 100
Recomendaciones únicas: 58


### **6.4 Reformulación**
1. Filtro por género
2. Agrupación de canciones repetidas
3. Distancia euclidiana
4. Similitud del coseno

In [16]:
# Eliminar canciones repetidas por título y artista
candidatas_unicas = (
    candidatas
    .assign(distancia_euclidiana = distancias)
    .sort_values(by = 'distancia_euclidiana', ascending = True)
    .drop_duplicates(subset = ['track_name', 'artists'], keep = 'first')
)

print('Candidatas originales:', len(candidatas))
print('Candidatas únicas:', len(candidatas_unicas))

Candidatas originales: 999
Candidatas únicas: 574


In [17]:
# Ordenar las candidatas por distancia euclidiana
candidatas_ordenadas = candidatas_unicas.sort_values(by = 'distancia_euclidiana', ascending = True)

# Seleccionar las 100 canciones más cercanas
df_candidatas = candidatas_ordenadas.head(100).copy()

# Recuperar sus índices
indices_seleccionados = df_candidatas.index

# Recuperar sus características musicales
X_candidatas = X_audio_scaled.loc[indices_seleccionados]

# Mostrar dimensiones
print('Canciones seleccionadas:', len(df_candidatas))
print('Dimensiones de X_candidatas:', X_candidatas.shape)

Canciones seleccionadas: 100
Dimensiones de X_candidatas: (100, 9)


In [18]:
# Calcular la similitud del coseno
similitudes = cosine_similarity(caracteristicas_referencia, X_candidatas).flatten()

# Asociar cada similitud con el índice de su canción
similitudes = pd.Series(similitudes, index = X_candidatas.index)

# Ordenar de mayor a menor similitud
similitudes_ordenadas = similitudes.sort_values(ascending = False)

In [19]:
# Construir la tabla de recomendaciones
recomendaciones = df_candidatas.loc[
    similitudes_ordenadas.index, ['track_name', 'artists', 'track_genre', 'album_name']
].copy()

# Incorporar las métricas
recomendaciones['distancia_euclidiana'] = distancias.loc[recomendaciones.index]

recomendaciones['similitud_coseno'] = similitudes_ordenadas

# Mostrar las 10 mejores recomendaciones
display(recomendaciones.head(10))

,track_name,artists,track_genre,album_name,distancia_euclidiana,similitud_coseno
83506,Rock And Roll All Nite,KISS,hard-rock;metal;rock,Rock Workout 2023,0.870047,0.976500
57512,Fly Away,Lenny Kravitz,rock,90's Music Grandi Successi,1.273074,0.933966
81453,My Songs Know What You Did In The Dark (Light ...,Fall Out Boy,emo;rock,Halloween Party 2022,1.587308,0.910548
73688,In Over My Head,grandson,alt-rock;alternative;rock,20's Rock,1.588413,0.909358
57259,"Sex, Drugs, Etc.",Beach Weather,rock,Chit Chat,1.890935,0.874359
83511,Hell Or High Water,KISS,hard-rock;metal;rock,Rock & Halloween,1.851981,0.863101
57275,Follow You,Imagine Dragons,rock,pov: it's 2021,2.193395,0.837489
84616,Good Time,Owl City;Carly Rae Jepsen,indie;indie-pop;rock,Tutti in piscina Canzoni per bambini,2.111448,0.832613
81986,Christmas All Over Again,Tom Petty and the Heartbreakers,folk;rock;singer-songwriter;songwriter,Rockin' Around The Christmas Tree 2022,2.138039,0.828829
57300,Radioactive,Imagine Dragons,rock,Night Visions,2.077368,0.811069


### **6.4.1 Prueba con otra canción**

In [20]:
# Buscar una canción en el conjunto de datos
display(
    df[
        df['track_name'].str.contains(
            "Strobe",
            case = False,
            na = False
        )
    ][['track_name', 'artists', 'track_genre', 'album_name']]
)

,track_name,artists,track_genre,album_name
81379,Strobe - Radio Edit,deadmau5,electronic;progressive-house,Strobe
81392,Strobe,deadmau5,electronic;progressive-house,Strobe


In [21]:
indice = 81392

# Recuperar sus géneros
generos_referencia = df.loc[indice, 'genres_set']

# Identificar canciones que compartan al menos un género
mascara_genero = df['genres_set'].apply(
    lambda generos: bool(generos_referencia & generos)
)

# Excluir la canción de referencia
mascara_genero.loc[indice] = False

# Recuperar las canciones candidatas
candidatas = df.loc[mascara_genero].copy()

print('Número de canciones candidatas:', len(candidatas))
display(candidatas[['track_name', 'artists', 'track_genre']].head(10))

Número de canciones candidatas: 1964


,track_name,artists,track_genre
20985,You Know You Like It,DJ Snake;AlunaGeorge,electronic
20986,Flight,Otnicka;ONEIL,electronic
20987,En Sadhai,6091;Chai Lenin,electronic
20988,Kanaavile - Re imagined,6091;Rakhooo,electronic
20989,Drama [Slowed],Maga;JVLA,electronic
20990,Flashbacks - Slowed,CRASPORE,electronic
20991,Memories,Otnicka,electronic
20992,Horizon,ARTBAT,electronic
20993,Floated By,Peter Cat Recording Co.,electronic
20994,U Are My High,DJ Snake,electronic


In [22]:
# Recuperar las características musicales de la referencia
caracteristicas_referencia = X_audio_scaled.loc[[indice]]

# Recuperar las características de las candidatas
caracteristicas_candidatas = X_audio_scaled.loc[candidatas.index]

# Calcular las distancias euclidianas
distancias = euclidean_distances(caracteristicas_referencia, caracteristicas_candidatas).flatten()

# Asociar las distancias con los índices
distancias = pd.Series(distancias, index = candidatas.index)

# Eliminar canciones repetidas por título y artista
candidatas_unicas = (
    candidatas
    .assign(distancia_euclidiana = distancias)
    .sort_values(by = 'distancia_euclidiana', ascending = True)
    .drop_duplicates(subset = ['track_name', 'artists'], keep = 'first')
)

print('Candidatas originales:', len(candidatas))
print('Candidatas únicas:', len(candidatas_unicas))

Candidatas originales: 1964
Candidatas únicas: 1811


In [23]:
# Ordenar las candidatas por distancia euclidiana
candidatas_ordenadas = candidatas_unicas.sort_values(by = 'distancia_euclidiana', ascending = True)

# Seleccionar las 100 canciones más cercanas
df_candidatas = candidatas_ordenadas.head(100).copy()

# Recuperar sus índices
indices_seleccionados = df_candidatas.index

# Recuperar sus características musicales
X_candidatas = X_audio_scaled.loc[indices_seleccionados]

print('Canciones seleccionadas:', len(df_candidatas))
print('Dimensiones de X_candidatas:', X_candidatas.shape)

Canciones seleccionadas: 100
Dimensiones de X_candidatas: (100, 9)


In [24]:
# Calcular la similitud del coseno
similitudes = cosine_similarity(caracteristicas_referencia, X_candidatas).flatten()

# Asociar las similitudes con los índices
similitudes = pd.Series(similitudes, index = X_candidatas.index)

# Ordenar de mayor a menor similitud
similitudes_ordenadas = similitudes.sort_values(ascending = False)

# Construir la tabla de recomendaciones
recomendaciones = df_candidatas.loc[
    similitudes_ordenadas.index,
    ['track_name', 'artists', 'track_genre', 'album_name']
].copy()

# Incorporar las métricas
recomendaciones['distancia_euclidiana'] = distancias.loc[recomendaciones.index]
recomendaciones['similitud_coseno'] = similitudes_ordenadas

# Mostrar las 10 mejores recomendaciones
display(recomendaciones.head(10))

,track_name,artists,track_genre,album_name,distancia_euclidiana,similitud_coseno
81312,La clé des champs,NTO,electronic;french;minimal-techno;techno,La clé des champs,0.662707,0.977742
21151,It's Ours,ARTBAT;David Guetta;Idris Elba,electronic,It's Ours,0.872574,0.961106
21470,Abu Simbel,WhoMadeWho,electronic,Abu Simbel,0.940711,0.957609
21449,Hellwalker,Mick Gordon,electronic,Doom (Original Game Soundtrack),1.105249,0.937419
54758,Project T - Martin Garrix Remix,Dimitri Vegas & Like Mike;Sander van Doorn;Mar...,progressive-house,Project T (Martin Garrix Remix),1.127264,0.934609
81316,Nopalitos,Kid Francescoli,electronic;french,Nopalitos,1.174544,0.933468
54655,Out Of Reach,Cristoph;Yotto;Sansa,progressive-house,Out Of Reach,1.193925,0.928888
54724,And We Knew It Was Our Time,Lane 8;Massane,progressive-house,Cross Pollination,1.171677,0.928478
21019,Bfg Division,Mick Gordon,electronic,Doom (Original Game Soundtrack),1.270546,0.925128
74741,Sexy Boy,Air,ambient;electronic;trip-hop,Moon Safari,1.371151,0.922577


### **6.4.2 Prueba con otra canción (múltiples géneros)**

In [25]:
# Buscar la canción con la mayor cantidad de géneros en el conjunto de datos
display(df[df['genres_set'].apply(len) == 9][['track_name', 'artists', 'track_genre', 'album_name']])

,track_name,artists,track_genre,album_name
75633,Baby Blue - Remastered 2010,Badfinger,blues;country;folk;j-pop;j-rock;power-pop;psyc...,Straight Up (Remastered 2010 / Deluxe Edition)


In [26]:
indice = 75633

# Recuperar sus géneros
generos_referencia = df.loc[indice, 'genres_set']

# Identificar canciones que compartan al menos un género
mascara_genero = df['genres_set'].apply(
    lambda generos: bool(generos_referencia & generos)
)

# Excluir la canción de referencia
mascara_genero.loc[indice] = False

# Recuperar las canciones candidatas
candidatas = df.loc[mascara_genero].copy()

print('Número de canciones candidatas:', len(candidatas))
display(candidatas[['track_name', 'artists', 'track_genre']].head(10))

Número de canciones candidatas: 7153


,track_name,artists,track_genre
5299,Prisoner,Raphael Lake;Aaron Levy;Daniel Ryan Murphy,blues
5300,I Shall Be Released,Joe Cocker,blues
5301,For the Glory,All Good Things,blues
5302,Run Rudolph Run,Chuck Berry,blues
5303,Run Rudolph Run,Chuck Berry,blues
5304,Feelin' Alright,Joe Cocker,blues
5305,Delta Lady,Joe Cocker,blues
5306,Cajun Moon,J.J. Cale,blues
5307,Ex's & Oh's,Elle King,blues
5308,Drunk (And I Don't Wanna Go Home),Elle King;Miranda Lambert,blues


In [27]:
# Recuperar las características musicales de la referencia
caracteristicas_referencia = X_audio_scaled.loc[[indice]]

# Recuperar las características de las candidatas
caracteristicas_candidatas = X_audio_scaled.loc[candidatas.index]

# Calcular las distancias euclidianas
distancias = euclidean_distances(caracteristicas_referencia, caracteristicas_candidatas).flatten()

# Asociar las distancias con los índices
distancias = pd.Series(distancias, index = candidatas.index)

# Eliminar canciones repetidas por título y artista
candidatas_unicas = (
    candidatas
    .assign(distancia_euclidiana = distancias)
    .sort_values(by = 'distancia_euclidiana', ascending = True)
    .drop_duplicates(subset = ['track_name', 'artists'], keep = 'first')
)

print('Candidatas originales:', len(candidatas))
print('Candidatas únicas:', len(candidatas_unicas))

Candidatas originales: 7153
Candidatas únicas: 5857


In [28]:
# Ordenar las candidatas por distancia euclidiana
candidatas_ordenadas = candidatas_unicas.sort_values(by = 'distancia_euclidiana', ascending = True)

# Seleccionar las 100 canciones más cercanas
df_candidatas = candidatas_ordenadas.head(100).copy()

# Recuperar sus índices
indices_seleccionados = df_candidatas.index

# Recuperar sus características musicales
X_candidatas = X_audio_scaled.loc[indices_seleccionados]

print('Canciones seleccionadas:', len(df_candidatas))
print('Dimensiones de X_candidatas:', X_candidatas.shape)

Canciones seleccionadas: 100
Dimensiones de X_candidatas: (100, 9)


In [29]:
# Calcular la similitud del coseno
similitudes = cosine_similarity(caracteristicas_referencia, X_candidatas).flatten()

# Asociar las similitudes con los índices
similitudes = pd.Series(similitudes, index = X_candidatas.index)

# Ordenar de mayor a menor similitud
similitudes_ordenadas = similitudes.sort_values(ascending = False)

# Construir la tabla de recomendaciones
recomendaciones = df_candidatas.loc[
    similitudes_ordenadas.index,
    ['track_name', 'artists', 'track_genre', 'album_name']
].copy()

# Incorporar las métricas
recomendaciones['distancia_euclidiana'] = distancias.loc[recomendaciones.index]
recomendaciones['similitud_coseno'] = similitudes_ordenadas

# Mostrar las 10 mejores recomendaciones
display(recomendaciones.head(10))

,track_name,artists,track_genre,album_name,distancia_euclidiana,similitud_coseno
85123,BAD FEELING,BOØWY,j-pop;j-rock;rockabilly,BOφWY,0.531570,0.965863
53761,Confetti - Remastered,The Lemonheads,power-pop,It's A Shame About Ray (30th Anniversary Edition),0.527115,0.952575
75054,Merry-Go-Round,MAN WITH A MISSION,anime;j-pop;j-rock,Merry-Go-Round,0.568596,0.949568
55058,California Dreamin' - 2007 Digital Remaster,The Beach Boys,psych-rock,The Warmth Of The Sun,0.607553,0.932263
75755,Sharp Dressed Man - 2008 Remaster,ZZ Top,blues;hard-rock,Eliminator,0.672586,0.925617
81876,Clarity,Vance Joy,folk;rock,Tek It - New Noise,0.696299,0.919850
74966,世界は恋に落ちている,CHiCO with HoneyWorks,anime;j-pop;j-rock,世界はiに満ちている,0.716232,0.919801
75019,ダブルシャッフル,Nana Mizuki,anime;j-pop,ダブルシャッフル,0.848430,0.919690
55110,California Dreamin',The Beach Boys,psych-rock,Stranger Things: Soundtrack from the Netflix S...,0.668918,0.917809
53731,Mary Anne - Remastered,Marshall Crenshaw,power-pop,Classic USA Pop,0.749255,0.915062


# **7. Construcción de la función definitiva**

In [30]:
def recomendar_canciones(indice, n_recomendaciones = 10):
    
    # Validar el índice de referencia
    if indice not in df.index:
        raise ValueError('El índice indicado no existe en el conjunto de datos.')

    # Validar el número de recomendaciones
    if not isinstance(n_recomendaciones, int) or isinstance(n_recomendaciones, bool):
        raise TypeError('El número de recomendaciones debe ser un entero.')

    if n_recomendaciones <= 0:
        raise ValueError('El número de recomendaciones debe ser positivo.')
    
    # Obtener los géneros de la canción de referencia
    generos_referencia = df.loc[indice, 'genres_set']

    # Filtrar canciones que compartan al menos un género
    mascara_genero = df['genres_set'].apply(
        lambda generos: bool(generos_referencia & generos)
    )

    # Excluir la canción de referencia
    mascara_genero.loc[indice] = False
    candidatas = df.loc[mascara_genero].copy()
    
    # Obtener las características acústicas
    caracteristicas_referencia = X_audio_scaled.loc[[indice]]
    caracteristicas_candidatas = X_audio_scaled.loc[candidatas.index]

    # Calcular las distancias euclidianas
    distancias = euclidean_distances(caracteristicas_referencia, caracteristicas_candidatas).flatten()
    distancias = pd.Series(distancias, index = candidatas.index)

    # Incorporar las distancias y eliminar canciones repetidas
    candidatas_unicas = (
        candidatas
        .assign(distancia_euclidiana = distancias)
        .sort_values(by = 'distancia_euclidiana', ascending = True)
        .drop_duplicates(subset = ['track_name', 'artists'], keep = 'first')
    )
    
    # Seleccionar las 100 canciones más cercanas
    candidatas_ordenadas = candidatas_unicas.sort_values(by = 'distancia_euclidiana', ascending = True)
    df_candidatas = candidatas_ordenadas.head(100).copy()

    indices_seleccionados = df_candidatas.index
    X_candidatas = X_audio_scaled.loc[indices_seleccionados]

    # Calcular la similitud del coseno
    similitudes = cosine_similarity(caracteristicas_referencia, X_candidatas).flatten()
    similitudes = pd.Series(similitudes, index = X_candidatas.index)

    # Ordenar por similitud del coseno
    similitudes_ordenadas = similitudes.sort_values(ascending = False)
    
    # Construir la tabla de recomendaciones
    recomendaciones = df_candidatas.loc[
        similitudes_ordenadas.index, ['track_name', 'artists', 'track_genre', 'album_name']
    ].copy()

    recomendaciones['distancia_euclidiana'] = (
        df_candidatas.loc[recomendaciones.index, 'distancia_euclidiana']
    )
    recomendaciones['similitud_coseno'] = similitudes_ordenadas

    # Devolver las recomendaciones solicitadas
    return recomendaciones.head(n_recomendaciones)

In [31]:
recomendaciones = recomendar_canciones(
    indice = 57411,
    n_recomendaciones = 10
)

display(recomendaciones)

,track_name,artists,track_genre,album_name,distancia_euclidiana,similitud_coseno
83506,Rock And Roll All Nite,KISS,hard-rock;metal;rock,Rock Workout 2023,0.870047,0.976500
57512,Fly Away,Lenny Kravitz,rock,90's Music Grandi Successi,1.273074,0.933966
81453,My Songs Know What You Did In The Dark (Light ...,Fall Out Boy,emo;rock,Halloween Party 2022,1.587308,0.910548
73688,In Over My Head,grandson,alt-rock;alternative;rock,20's Rock,1.588413,0.909358
57259,"Sex, Drugs, Etc.",Beach Weather,rock,Chit Chat,1.890935,0.874359
83511,Hell Or High Water,KISS,hard-rock;metal;rock,Rock & Halloween,1.851981,0.863101
57275,Follow You,Imagine Dragons,rock,pov: it's 2021,2.193395,0.837489
84616,Good Time,Owl City;Carly Rae Jepsen,indie;indie-pop;rock,Tutti in piscina Canzoni per bambini,2.111448,0.832613
81986,Christmas All Over Again,Tom Petty and the Heartbreakers,folk;rock;singer-songwriter;songwriter,Rockin' Around The Christmas Tree 2022,2.138039,0.828829
57300,Radioactive,Imagine Dragons,rock,Night Visions,2.077368,0.811069


### **7.1 Búsqueda**

In [32]:
def buscar_cancion(nombre, artista = None, limite = 20):
    
    # Buscar coincidencias en el título
    mascara_nombre = df['track_name'].str.contains(nombre.strip(), case = False, na = False, regex = False)

    # Si se proporciona un artista, aplicar también ese filtro
    if artista is not None:
        mascara_artista = df['artists'].str.contains(artista.strip(), case = False, na = False, regex = False)
        coincidencias = df[mascara_nombre & mascara_artista]

    else:
        coincidencias = df[mascara_nombre]

    # Evaluar el número de coincidencias
    numero = len(coincidencias)

    if numero == 0:
        print('No se encontraron coincidencias.')

    elif numero == 1:
        print('Se encontró una coincidencia:')
        display(coincidencias[['track_name', 'artists', 'album_name']])

    else:
        print(f'Se encontraron {numero} coincidencias.')

        if numero > limite:
            print(f'Mostrando las primeras {limite} coincidencias.')

        display(coincidencias[['track_name', 'artists', 'album_name']].head(limite))

    return coincidencias

### **7.2 Selección**

In [33]:
def seleccionar_cancion():
    
    while True:

        # Solicitar los datos de búsqueda
        nombre = input('Introduce el nombre de la canción (o "salir" para terminar): ')

        # Comprobar si el usuario desea salir
        if nombre.strip().lower() in ['salir', 'exit']:
            print('Búsqueda finalizada.')
            return None

        # Comprobar que el nombre no esté vacío
        if not nombre.strip():
            print('Debes introducir el nombre de una canción.\n')
            continue

        artista = input('Introduce el artista (opcional): ')

        # Buscar coincidencias
        if artista.strip():
            coincidencias = buscar_cancion(nombre, artista)
        else:
            coincidencias = buscar_cancion(nombre)

        # Comprobar si existen coincidencias
        if coincidencias.empty:
            print('Intenta realizar una nueva búsqueda.\n')
            continue

        # Si existe una sola coincidencia, obtener su índice
        if len(coincidencias) == 1:
            indice = coincidencias.index[0]

        # Si existen varias coincidencias, solicitar una selección
        else:
            # Limitar las opciones a las primeras 20 coincidencias
            opciones = coincidencias.head(20)

            print('\nIntroduce el índice de la canción que deseas seleccionar.')

            while True:
                entrada = input('Índice: ')

                try:
                    indice = int(entrada)

                    if indice in opciones.index:
                        break
                    else:
                        print('El índice no corresponde a ninguna de las opciones mostradas.')

                except ValueError:
                    print('¡Debes introducir un índice numérico, zoquete!.')

        # Mostrar la canción seleccionada
        print('\nCanción seleccionada:')
        display(df.loc[[indice], ['track_name', 'artists', 'album_name']])

        # Solicitar confirmación
        while True:
            respuesta = input('¿Es esta la canción que buscabas? (sí/no): ')
            respuesta = respuesta.strip().lower()

            if respuesta in ['s', 'si', 'sí', 'y', 'yes']:
                print('\nCanción confirmada.')
                return indice

            elif respuesta in ['n', 'no']:
                print('\nVamos a realizar una nueva búsqueda.\n')
                break

            else:
                print('¡Respuesta no válida! Introduce "sí" o "no".')

In [34]:
indice_seleccionado = seleccionar_cancion()

print('Índice devuelto:', indice_seleccionado)

Se encontró una coincidencia:


,track_name,artists,album_name
57411,Don't Stop Me Now - Remastered 2011,Queen,Jazz (2011 Remaster)



Canción seleccionada:


,track_name,artists,album_name
57411,Don't Stop Me Now - Remastered 2011,Queen,Jazz (2011 Remaster)



Canción confirmada.
Índice devuelto: 57411


### **7.3 Función principal**

In [35]:
def ejecutar_recomendador(n_recomendaciones = 10):
    
    # Seleccionar una canción
    indice_seleccionado = seleccionar_cancion()

    # Generar recomendaciones si se confirmó una canción
    if indice_seleccionado is not None:

        recomendaciones = recomendar_canciones(
            indice = indice_seleccionado,
            n_recomendaciones = n_recomendaciones
        )

        print('\nCanciones recomendadas:')
        display(recomendaciones)

        return recomendaciones

    return None

In [36]:
recomendaciones = ejecutar_recomendador()

Se encontró una coincidencia:


,track_name,artists,album_name
57411,Don't Stop Me Now - Remastered 2011,Queen,Jazz (2011 Remaster)



Canción seleccionada:


,track_name,artists,album_name
57411,Don't Stop Me Now - Remastered 2011,Queen,Jazz (2011 Remaster)



Canción confirmada.

Canciones recomendadas:


,track_name,artists,track_genre,album_name,distancia_euclidiana,similitud_coseno
83506,Rock And Roll All Nite,KISS,hard-rock;metal;rock,Rock Workout 2023,0.870047,0.976500
57512,Fly Away,Lenny Kravitz,rock,90's Music Grandi Successi,1.273074,0.933966
81453,My Songs Know What You Did In The Dark (Light ...,Fall Out Boy,emo;rock,Halloween Party 2022,1.587308,0.910548
73688,In Over My Head,grandson,alt-rock;alternative;rock,20's Rock,1.588413,0.909358
57259,"Sex, Drugs, Etc.",Beach Weather,rock,Chit Chat,1.890935,0.874359
83511,Hell Or High Water,KISS,hard-rock;metal;rock,Rock & Halloween,1.851981,0.863101
57275,Follow You,Imagine Dragons,rock,pov: it's 2021,2.193395,0.837489
84616,Good Time,Owl City;Carly Rae Jepsen,indie;indie-pop;rock,Tutti in piscina Canzoni per bambini,2.111448,0.832613
81986,Christmas All Over Again,Tom Petty and the Heartbreakers,folk;rock;singer-songwriter;songwriter,Rockin' Around The Christmas Tree 2022,2.138039,0.828829
57300,Radioactive,Imagine Dragons,rock,Night Visions,2.077368,0.811069
